# V037 conditional final consumer proof

Missing/partial F evidence cannot start final fitting. Check exact capacity factory state/RNG/mean/scale, all-training loader contract, native postprocessing, and inherited strict objective/resume tests using synthetic data only.

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,tempfile,copy,os,gc
import numpy as np
import torch

def run_final_preflight_v037(project_root,output_directory,device='cpu'):
    P=Path(project_root).resolve();O=Path(output_directory);O.mkdir(parents=True,exist_ok=False);checks={}
    def code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
    def check(name,c):
        if not bool(c):raise AssertionError(name)
        checks[name]=True
    source=code(P/'final_fit_v037.ipynb');ns={'__name__':'final_fit_v037'};saved=os.environ.get('PA3_RUN_FINAL_V037');os.environ['PA3_RUN_FINAL_V037']='0'
    try:exec(compile(source,'final_fit_v037.ipynb','exec'),ns)
    finally:
        if saved is None:os.environ.pop('PA3_RUN_FINAL_V037',None)
        else:os.environ['PA3_RUN_FINAL_V037']=saved
    check('default_import_defines_final_consumer',callable(ns['run_final_v037']))
    with tempfile.TemporaryDirectory() as td:
        root=Path(td)
        for label,result_sha,audit_sha in [('both_missing',None,None),('missing_audit','a'*64,None),('wrongly_claimed_files','a'*64,'b'*64)]:
            try:ns['v037_confirmation_gate'](root,result_sha,audit_sha)
            except (ValueError,FileNotFoundError):checks['reject_before_data:'+label]=True
            else:raise AssertionError('Missing reviewed F gate accepted')
        check('gate_rejections_created_no_training_artifacts',list(root.iterdir())==[])
    for name in ['feature_components_v005.ipynb','sequence_components_v005.ipynb','training_components_v008.ipynb','laplace_components_v029.ipynb','postprocessing_components_v016.ipynb']:
        exec(compile(code(P/'research'/name),name,'exec'),ns)
    torch.set_num_threads(4);selected=ns['_sequence_device'](device);spec=json.loads((P/'research/confirmation_selection_v036.json').read_text())['required_members']['capacity_laplace_curve']['model_spec'];check('exact_frozen_capacity_spec',spec['widths']==[320,256,192,128] and spec['layers']==4)
    ns['seed_sequence'](20261102);original=ns['v029_laplace_factory'](spec,47,{}).to(selected);state=torch.get_rng_state().clone();cs=torch.cuda.get_rng_state_all() if device=='cuda' else None
    ns['seed_sequence'](20261102);wrapped=ns['v037_laplace_factory'](spec,47,{}).to(selected)
    check('final_factory_state_exact_selected_factory',all(torch.equal(v,wrapped.state_dict()[k]) for k,v in original.state_dict().items()));check('final_factory_CPU_RNG_exact',torch.equal(state,torch.get_rng_state()))
    if cs is not None:check('final_factory_CUDA_RNG_exact',all(torch.equal(a,b) for a,b in zip(cs,torch.cuda.get_rng_state_all())))
    raw=torch.randn(2,80,48,device=selected);raw[:,:,-1]=0;original.eval();wrapped.eval()
    with torch.no_grad():a=original(raw);b=wrapped(raw)
    check('final_factory_mu_exact_selected_factory',torch.equal(a,b));check('final_factory_scale_exact_selected_factory',torch.equal(original.v029_scale,wrapped.v029_scale));check('full_architecture_parameter_count',sum(p.numel() for p in wrapped.parameters())==5108162)
    # Confirm production helper dictionary matches the authoritative all-training
    # loader contract before any real data is read.
    import ast
    helper=code(P/'final_fit_v011.ipynb');tree=ast.parse(helper);loader=next(f for f in tree.body if isinstance(f,ast.FunctionDef) and f.name=='v011_load_all_training');returned=next(n for n in ast.walk(loader) if isinstance(n,ast.Return));check('all_training_loader_names_key_supported',isinstance(returned.value,ast.Call) and any(k.arg=='names' for k in returned.value.keywords) and "data['feature_names']" not in source)
    # Every member is projected with the same fixed policy; expiration preserves
    # a finite mean over all six members, and all input row IDs remain in order.
    weights=dict(original_compact=.125,curve=.125,memory=.125,offset=.125,deep=.25,capacity_laplace_curve=.25);pred={k:np.array([-1.2,.3,4.7,.9],np.float64)+i*.01 for i,k in enumerate(weights)};grid=np.array([-2.,-1.,0.,1.,2.,4.,5.]);phase=np.array([0,1,0,1]);mean=sum(weights[k]*pred[k] for k in weights);processed=ns['selected_postprocessing_v016'](pred,weights,grid);final=np.where(phase==0,processed,mean)
    check('final_policy_all_rows_finite',final.shape==(4,) and np.isfinite(final).all());check('expiration_fixed_mean_exact',np.array_equal(final[phase==1],mean[phase==1]));check('inspiration_training_grid_exact',np.array_equal(final[phase==0],processed[phase==0]))
    check('final_source_requires_complete_not_stage_audit',"audit['status']=='passed'" in source and 'Seven cold F members' in source and "len(result['records'])==7" in source)
    check('final_inference_negative_values_not_clipped',final[0]<0)
    del original,wrapped;gc.collect()
    # Reuse complete objective/masking/native-inference/strict-resume tests of
    # this same Laplace backbone on the requested actual device.
    proof_source=code(P/'research/laplace_preflight_v029.ipynb');check('inherited_objective_proof_pinned',hashlib.sha256(proof_source.encode()).hexdigest()=='4143266b792b6d6eb19fa1437d755a2f5685abdfe4ccfaaeeab1a7bd7743aca3');extra={};exec(compile(proof_source,'laplace_preflight_v029.ipynb','exec'),extra);inherited=extra['run_laplace_preflight_v029'](P,O/'inherited_laplace',device=device,deep_spec=spec);checks.update({'inherited_'+k:v for k,v in inherited['checks'].items()})
    result=dict(status='passed',created_utc=datetime.now(timezone.utc).isoformat(),check_count=len(checks),checks=checks,final_code_sha256=hashlib.sha256(source.encode()).hexdigest(),actual_cuda=device=='cuda',no_F_outcomes_read=True,no_approved_data_read=True,no_final_or_assignment_fits=True,synthetic_model_fits_only=True,actual_all_data_model_reuse_not_run=True,actual_F_gate_not_passed=True,runtime=dict(torch=str(torch.__version__),numpy=np.__version__,cuda=torch.version.cuda,device=device))
    (O/'proof.json').write_text(json.dumps(result,indent=2)+'\n');print(json.dumps(result,indent=2));return result
